In [1]:
import pandas as pd

In [2]:
threads = pd.read_parquet("data/threads.parquet")
submissions = pd.read_parquet("data/submissions.parquet")

In [4]:
threads.head(10)

,comm_id,sub_id,parent_id,subreddit_id,author_id,created_at,predicted_stance_scores,subreddit,thread_id
0,comm71631,t3_sub3423,t1_comm71632,r/4,author26899,2021-02-22 04:26:38,NaN,subreddit4,lowvyz
1,comm71633,t3_sub3423,t1_comm71634,r/4,author26899,2021-02-22 04:25:20,0.0,subreddit4,lowvyz
2,comm71635,t3_sub3423,t1_comm71636,r/4,author19952,2021-02-21 20:33:18,NaN,subreddit4,lowvyz
3,comm71634,t3_sub3423,t1_comm71637,r/4,author26900,2021-02-21 18:09:22,0.5,subreddit4,lowvyz
4,comm71636,t3_sub3423,t3_sub3423,r/4,author1577,2021-02-21 16:49:42,0.0,subreddit4,lowvyz
5,comm71638,t3_sub3423,t3_sub3423,r/4,author458,2021-02-21 16:30:23,0.0,subreddit4,lowvyz
6,comm71639,t3_sub3423,t3_sub3423,r/4,author26217,2021-02-21 16:25:18,0.0,subreddit4,lowvyz
7,comm71640,t3_sub3423,t3_sub3423,r/4,author20722,2021-02-21 16:17:39,0.0,subreddit4,lowvyz
8,comm71637,t3_sub3423,t1_comm71641,r/4,author26217,2021-02-21 16:15:13,0.0,subreddit4,lowvyz
9,comm71642,t3_sub3423,t1_comm71643,r/4,author20727,2021-02-21 15:06:49,0.5,subreddit4,lowvyz


In [3]:
# Creating nodes and opinions table
# All items
keep = ["subreddit_id", "item_id", "sub_id", "kind", "author_id", "created_at", "predicted_stance_scores"]
comments = threads.assign(item_id=threads["comm_id"], kind="comment",
                          sub_id=threads["sub_id"].str.replace("t3_", "", regex=False))
sub = submissions.assign(item_id=submissions["sub_id"], kind="submission")
items = pd.concat([sub[keep], comments[keep]], ignore_index=True)
items["t"] = pd.to_datetime(items["created_at"])
items["score"] = pd.to_numeric(items["predicted_stance_scores"], errors="coerce")
items = items.dropna(subset=["author_id"])

# Opinions Dataset
opinions = (items.dropna(subset=["score"])
            [["author_id", "t", "score", "kind", "subreddit_id", "sub_id", "item_id"]]
            .sort_values(["author_id", "t"]).reset_index(drop=True))

# Nodes dataset
nodes = items.groupby("author_id").agg(n_items=("item_id", "size"),
                                       n_subreddits=("subreddit_id", "nunique"),
                                       first_t=("t", "min"), last_t=("t", "max"))
op = opinions.groupby("author_id")["score"]
nodes["n_scored"] = op.size()
nodes["first_score"], nodes["last_score"] = op.first(), op.last()
nodes["n_scored"] = nodes["n_scored"].fillna(0).astype(int)
nodes = nodes.reset_index()
nodes["node_index"] = range(len(nodes))

print(nodes.shape, opinions.shape)
print("users with >=1 scored item:", (nodes.n_scored >= 1).sum(), "| >=2:", (nodes.n_scored >= 2).sum())

(54928, 9) (171655, 7)
users with >=1 scored item: 43861 | >=2: 25294


In [5]:
(nodes.n_subreddits > 1).mean()

np.float64(0.09608942615787941)

In [4]:
nodes.head(10)

,author_id,n_items,n_subreddits,first_t,last_t,n_scored,first_score,last_score,node_index
0,author0,3,1,2022-10-12 22:45:46,2022-10-13 00:34:27,1,0.50,0.50,0
1,author1,5,1,2022-09-26 16:01:46,2022-09-28 14:44:18,0,NaN,NaN,1
2,author10,4,1,2022-02-03 19:33:09,2022-07-24 20:11:41,2,0.75,0.00,2
3,author100,2,2,2020-09-13 07:25:34,2022-07-10 03:22:48,2,0.75,0.90,3
4,author1000,1,1,2021-07-25 01:04:20,2021-07-25 01:04:20,1,0.50,0.50,4
5,author10000,2,1,2020-12-17 07:27:30,2022-10-20 23:17:14,1,0.00,0.00,5
6,author10001,2,1,2022-10-20 21:01:00,2022-10-20 23:16:49,2,0.00,0.50,6
7,author10002,1,1,2022-10-20 22:39:58,2022-10-20 22:39:58,1,0.50,0.50,7
8,author10003,4,1,2022-10-20 14:58:42,2022-10-20 21:55:35,2,0.90,0.86,8
9,author10004,2,1,2022-10-20 13:27:35,2022-10-20 21:48:12,1,0.50,0.50,9


In [6]:
opinions.head(10)

,author_id,t,score,kind,subreddit_id,sub_id,item_id
0,author0,2022-10-12 23:22:49,0.50,comment,r/0,sub0,comm43201
1,author10,2022-05-17 15:40:37,0.75,comment,r/0,sub20,comm40516
2,author10,2022-07-24 20:11:41,0.00,submission,r/0,sub11,sub11
3,author100,2020-09-13 07:25:34,0.75,submission,r/0,sub123,sub123
4,author100,2022-07-10 03:22:48,0.90,comment,r/2,sub609,comm190288
5,author1000,2021-07-25 01:04:20,0.50,submission,r/2,sub1693,sub1693
6,author10000,2022-10-20 23:17:14,0.00,comment,r/3,sub2763,comm15350
7,author10001,2022-10-20 21:01:00,0.00,comment,r/3,sub2763,comm15362
8,author10001,2022-10-20 23:16:49,0.50,comment,r/3,sub2763,comm15352
9,author10002,2022-10-20 22:39:58,0.50,comment,r/3,sub2763,comm15356
